# Project 4 - Neural Radiance Field (NeRF)

In [ ]:
# load libraries
import numpy as np
import skimage as sk
import skimage.io as skio
from skimage import img_as_ubyte
import matplotlib.pyplot as plt
from skimage.transform import resize

import cv2
import numpy as np
import glob # for loadig multiple files

## Part 0: Calibrating Your Camera and Capturing a 3D Scan

### Part 0.1: Calibrating Your Camera

Loop through all your calibration images
For each image, detect the ArUco tags using OpenCV's ArUco detector
Extract the corner coordinates from the detected tags
Collect all detected corners and their corresponding 3D world coordinates (you can consider the ArUco tag as the world origin and define the 4 corners' 3D points relative to that, e.g., if your tag is 0.02m × 0.02m, the corners could be [(0,0,0), (0.02,0,0), (0.02,0.02,0), (0,0.02,0)])
Use cv2.calibrateCamera() to compute the camera intrinsics and distortion coefficients

In [ ]:
# calibration pipeline

# starter code from spec
# import cv2
# import numpy as np

# Create ArUco dictionary and detector parameters (4x4 tags)
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
aruco_params = cv2.aruco.DetectorParameters()

# tag size
tag_size = 0.02 # meters, 0.02 = 2 cm
# 3d coordinates of aruco tag in world frame
obj_pts_single = np.array([
    [0, 0, 0],           # Top-left
    [tag_size, 0, 0],    # Top-right
    [tag_size, tag_size, 0],  # Bottom-right
    [0, tag_size, 0]     # Bottom-left
], dtype = np.float32)

obj_pts = [] # all 3D pts in world frame
img_pts = [] # all 2D pts in image frame

images = glob.glob('path_to_your_images/*.jpg') # path to images

for path in images:
    # load image
    im = cv2.imread(path)
    gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)

    # Detect ArUco markers in an image
    # Returns: corners (list of numpy arrays), ids (numpy array)
    corners, ids, _ = cv2.aruco.detectMarkers(gray, aruco_dict, parameters=aruco_params) # modify to take in grayscale image

    # Check if any markers were detected
    if ids is not None:
    # Process the detected corners
    # corners: list of length N (number of detected tags)
    #   - each element is a numpy array of shape (1, 4, 2) containing the 4 corner coordinates (x, y)
    # ids: numpy array of shape (N, 1) containing the tag IDs for each detected marker
    # Example: if 3 tags detected, corners will be a list of 3 arrays, ids will be shape (3, 1)
        # loop through all detected markers
        for c in corners:
            img_pts.append(c.reshape(-1, 2)) # reshape from (1, 4, 2) to shape: (4,2)
            obj_pts.append(obj_pts_single) # append the same 3D points for each detected marker
    else:
        # No tags detected in this image, skip it
        print(f"No ArUco detected in {path}, skipping.")

# 5. calibrate camera
if len(obj_pts > 0):
    img_shape = gray.shape[::-1] # (width, height)
    
    ret, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
        obj_pts,
        img_pts,
        img_shape,
        None, # initial camera matrix (estimated by OpenCV)
        None # initial distortion coefficients
    )

    print("Calibration successful!")
    print("Camera matrix (intrinsics):\n", camera_matrix)
    print("Distortion coefficients:\n", dist_coeffs)
else:
    print("No valid markers detected in any image. Calibration unsuccessful.")
